# 05: Measure your circuit on a quantum computer

**What you will learn:** ideal probabilities describe a model; real hardware returns finite-shot counts. A shot is one execution followed by measurement.

We will send a Bell circuit to Quafu and compare its counts with the ideal prediction.


## 1. Prepare the workshop

Run setup unchanged. Platform credentials are read automatically, never entered or displayed here. The saved result will also be used by notebook 06.


In [ ]:
from pathlib import Path
import sys

search_roots = (Path.cwd().resolve(), *Path.cwd().resolve().parents)
WORKSHOP = next((root / "flagos2026" for root in search_roots
                 if (root / "flagos2026/notebooks").is_dir()), None)
if WORKSHOP is None:
    raise RuntimeError("Cannot locate flagos2026/notebooks. Open the deployed workshop first.")
sys.path.insert(0, str(WORKSHOP / "scripts"))
from cloud_lesson import prepare_lesson

lesson = prepare_lesson(Path.cwd(), "quafu-bell")
WORKSHOP = lesson.workshop
repository = WORKSHOP.parent.parent
ROOT = repository if (repository / "pyproject.toml").is_file() else WORKSHOP
OUTPUTS = lesson.outputs


## 2. Build the circuit and check the ideal answer

Ideally only `00` and `11` appear, each with probability one half.


In [ ]:
import flagquantum as fq

circuit = fq.Circuit(2).h(0).cx(0, 1)
ideal = fq.run(circuit)
print("Ideal probabilities:", ideal.state.abs().square().tolist())


## 3. Submit one real hardware task

Confirm today’s available target with your instructor, then enable the switch. **This creates one real QPU task with 1024 shots and uses your hardware quota.**

Quafu’s task service compiles the logical circuit for the device. The workshop shortcut calls `fq.submit(..., compiler=None)` and saves the receipt immediately; it does not wait for the measurement to finish. Rerunning reconnects to the same task.


In [ ]:
target = "quafu:Dongling"
shots = 1024
submit_to_hardware = False

job = lesson.submit_once(
    circuit, target=target, shots=shots, enabled=submit_to_hardware,
)
if job is not None:
    print("Task ID:", job.id)


## 4. Check progress

This is a read-only query. A queued task has already been submitted; it should not be submitted again.


In [ ]:
if job is not None:
    print("Status:", job.status())


## 5. Read the measurement counts

Rerun this result cell later if the task is still pending. Completed counts are automatically saved for notebook 06.


In [ ]:
result = lesson.collect(job)
if result is not None:
    counts = result.counts[0]
    print("Measured counts:", counts)
    assert sum(counts.values()) == shots
    print("Total shots:", sum(counts.values()))


## Interpret the result

- Are `01` and `10` completely absent? What could cause them to appear?
- Would twice as many shots necessarily remove hardware noise?
- What is the difference between submitting, waiting and receiving a completed result?

Continue to notebook 06 to quantify the difference. For receipt recovery and standalone API examples, see [Cloud job field notes](../../CLOUD_JOBS.md).
